In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
import random

# Set random seed for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # Set a fixed seed for reproducibility

# Set device
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

# Load pre-trained RegNetY-32GF
model = models.regnet_y_32gf(weights=models.RegNet_Y_32GF_Weights.IMAGENET1K_V2)

# Modify classifier
num_features = model.fc.in_features
num_classes = 100  # Change this according to your dataset
model.fc = nn.Linear(num_features, num_classes)
model = model.to(device)

# Validation transform
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(root='../train/train', transform=transforms.ToTensor())
train_size = int(0.8 * len(full_dataset))  # 80% for training
val_size = len(full_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Apply different transforms to validation dataset
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    set_seed(42 + epoch)  # Change seed slightly each epoch for variation
    
    # Define new random data augmentation for each epoch
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    train_dataset.dataset.transform = train_transform
    
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Validation Accuracy: {100 * correct / total:.2f}%")

# Save model
torch.save(model.state_dict(), './weights/regnet_transfer.pth')
print("Model saved as regnet_transfer.pth")

torch_script_model = torch.jit.script(model)
torch.jit.save(torch_script_model,'./model_script/regnet_transfer.pth')


Epoch 1, Loss: 4.2705
Validation Accuracy: 10.00%
Epoch 2, Loss: 2.7591
Validation Accuracy: 22.00%
Epoch 3, Loss: 2.0221
Validation Accuracy: 24.00%
Epoch 4, Loss: 1.5476
Validation Accuracy: 27.50%
Epoch 5, Loss: 1.2288
Validation Accuracy: 30.50%
Epoch 6, Loss: 1.0092
Validation Accuracy: 33.50%
Epoch 7, Loss: 0.8363
Validation Accuracy: 33.00%
Epoch 8, Loss: 0.7215
Validation Accuracy: 34.50%
Epoch 9, Loss: 0.5987
Validation Accuracy: 38.50%
Epoch 10, Loss: 0.5475
Validation Accuracy: 38.00%
Epoch 11, Loss: 0.4885
Validation Accuracy: 39.00%
Epoch 12, Loss: 0.4401
Validation Accuracy: 37.00%
Epoch 13, Loss: 0.3923
Validation Accuracy: 38.50%
Epoch 14, Loss: 0.3653
Validation Accuracy: 38.50%
Epoch 15, Loss: 0.3210
Validation Accuracy: 38.00%
Epoch 16, Loss: 0.3068
Validation Accuracy: 41.00%
Epoch 17, Loss: 0.2863
Validation Accuracy: 36.00%
Epoch 18, Loss: 0.2633
Validation Accuracy: 37.00%
Epoch 19, Loss: 0.2472
Validation Accuracy: 37.50%
Epoch 20, Loss: 0.2267
Validation Accura

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
import random

# Set random seed for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # Set a fixed seed for reproducibility

# Set device
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

# Load pre-trained RegNetY-32GF
model = models.regnet_y_32gf(weights=models.RegNet_Y_32GF_Weights.DEFAULT)

# Modify classifier
num_features = model.fc.in_features
num_classes = 100  # Change this according to your dataset
model.fc = nn.Linear(num_features, num_classes)
model = model.to(device)

# Validation transform
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(root='../train/train', transform=transforms.ToTensor())
train_size = int(0.8 * len(full_dataset))  # 80% for training
val_size = len(full_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Apply different transforms to validation dataset
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Unfreeze more layers for fine-tuning
for param in model.parameters():
    param.requires_grad = True

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    set_seed(42 + epoch)  # Change seed slightly each epoch for variation
    
    # Define new random data augmentation for each epoch
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.AutoAugment(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    train_dataset.dataset.transform = train_transform
    
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Validation Accuracy: {100 * correct / total:.2f}%")

# Save model
torch.save(model.state_dict(), './weights/regnet_transfer2.pth')
print("Model saved as regnet_transfer.pth")

torch_script_model = torch.jit.script(model)
torch.jit.save(torch_script_model,'./model_script/regnet_transfer2.pth')


Epoch 1, Loss: 4.5007
Validation Accuracy: 11.00%
Epoch 2, Loss: 3.5674


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
import random

# Set random seed for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # Set a fixed seed for reproducibility

# Set device
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

# Load dataset
full_dataset = datasets.ImageFolder(root='../train/train', transform=transforms.ToTensor())
num_classes = len(full_dataset.classes)  # Automatically detect number of classes

# Load pre-trained RegNetY-32GF
model = models.regnet_y_32gf(weights=models.RegNet_Y_32GF_Weights.DEFAULT)

# Modify classifier
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)  # Adjust to detected class count
model = model.to(device)

# Validation transform
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_size = int(0.8 * len(full_dataset))  # 80% for training
val_size = len(full_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Apply different transforms to validation dataset
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Loss function and optimizer with weight decay (L2 regularization)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Add Dropout and BatchNorm layers to the model
for name, module in model.named_children():
    if isinstance(module, nn.Linear):
        model.add_module(f"{name}_dropout", nn.Dropout(0.5))

# Unfreeze more layers for fine-tuning
for param in model.parameters():
    param.requires_grad = True

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# Training loop
num_epochs = 50
best_val_acc = 0.0  # Track best validation accuracy for early stopping
for epoch in range(num_epochs):
    set_seed(42 + epoch)  # Change seed slightly each epoch for variation
    
    # Define new random data augmentation for each epoch
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.AutoAugment(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    train_dataset.dataset.transform = train_transform
    
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_acc = 100 * correct / total
    print(f"Validation Accuracy: {val_acc:.2f}%")

    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'regnet_transfer.pth')
        print("Model saved as regnet_transfer.pth")

    # Update learning rate
    scheduler.step()

print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

# Save model
torch.save(model.state_dict(), './weights/regnet_transfer3.pth')
print("Model saved as regnet_transfer.pth")

torch_script_model = torch.jit.script(model)
torch.jit.save(torch_script_model,'./model_script/regnet_transfer3.pth')